In [1]:
from pathlib import Path

import pandas as pd
from tqdm.auto import tqdm
from rdkit import Chem
from rdkit.Chem import PandasTools
from rdkit.Chem.FilterCatalog import FilterCatalog, FilterCatalogParams

In [2]:
#01 Menentukan direktori kerja saat ini
HERE = Path.cwd()
DATA = HERE / "data"

In [ ]:
#02 Memuat data dari hasil filtrasi Ro5
ligandsLipinski = pd.read_csv(
    DATA / "02NPlipinskiFulfilled.csv",
    index_col=0,
)
# Menghapus informasi yang tidak diperlukan
print("Ukuran DataFrame:", ligandsLipinski.shape)
ligandsLipinski.drop(columns=["molecular_weight", "n_hbd", "n_hba", "logp"], inplace=True)
ligandsLipinski.head(10)

In [ ]:
#03 Menambahkan kolom molekul
PandasTools.AddMoleculeColumnToFrame(ligandsLipinski, smilesCol="smiles")
# Menggambar 9 molekul pertama
Chem.Draw.MolsToGridImage(
    list(ligandsLipinski.head(9).ROMol),
    legends=list(ligandsLipinski.head(9).pref_name),
)

In [6]:
#04 Inisialisasi filter menggunakan rdkit.Chem.FilterCatalog
params = FilterCatalogParams()
params.AddCatalog(FilterCatalogParams.FilterCatalogs.PAINS)
params.AddCatalog(FilterCatalogParams.FilterCatalogs.BRENK)
params.AddCatalog(FilterCatalogParams.FilterCatalogs.NIH)
catalog = FilterCatalog(params)

In [7]:
#05 Pencarian untuk substructure PAINS, BRENK, dan NIH
matches = []
clean = []
for index, row in tqdm(ligandsLipinski.iterrows(), total=ligandsLipinski.shape[0]):
    molecule = Chem.MolFromSmiles(row.smiles)
    entry = catalog.GetFirstMatch(molecule)  # Mendapatkan kecocokan pertama untuk PAINS, BRENK, dan NIH
    if entry is not None:
        # Menyimpan informasi PAINS, BRENK, dan NIH
        matches.append(
            {
                "molecule_chembl_id": row.molecule_chembl_id,
                "rdkit_molecule": molecule,
                "pains_brenk_NIH": entry.GetDescription().capitalize(),
            }
        )
    else:
        # Mengumpulkan indeks molekul tanpa PAINS, BRENK, dan NIH
        clean.append(index)

matches = pd.DataFrame(matches)
matches = matches.merge(ligandsLipinski, left_on='molecule_chembl_id', right_on='molecule_chembl_id')
ligandsFiltered = ligandsLipinski.loc[clean]  # Menyimpan molekul tanpa PAINS, BRENK, dan NIH

  0%|          | 0/155 [00:00<?, ?it/s]

In [8]:
#06 Cek jumlah molekul yang telah terfiltrasi
print(f"Jumlah senyawa dengan PAINS, BRENK, dan NIH: {len(matches)}")
print(f"Jumlah senyawa tanpa PAINS, BRENK, dan NIH: {len(ligandsFiltered)}")

Jumlah senyawa dengan PAINS, BRENK, dan NIH: 101
Jumlah senyawa tanpa PAINS, BRENK, dan NIH: 54


In [ ]:
#07 Cek struktur molekul yang telah terfiltrasi
legends = (
    matches.head(8)
    .apply(lambda x: f"{x['pains_brenk_NIH']}\n {x['pref_name']}", axis=1)
    .tolist()
)
#Memeriksa molekul yang terfiltrasi dengan PAINS, BRENK, dan NIH
Chem.Draw.MolsToGridImage(
    list(matches.head(8).rdkit_molecule),
    legends=legends,
)

In [ ]:
#08 Memeriksa data yang sudah difilter
ligandsFiltered.reset_index(drop=True, inplace=True)
ligandsFiltered.head()

In [12]:
#09 Menyimpan data yang sudah difilter ke dalam file CSV
ligandsFiltered.to_csv(DATA / "03NPstructureFiltered.csv")